# arms_embed_target — similarity to the target on held-out rows

`arms_embed` measured how close each arm's output sits to the row's reference images. This
measures how close it sits to the row's target, on the 488 test rows, where the target was never
seen in training.

The positive control established the memorised case: on rows the adapter trained on, more training
moved the output toward the target (DINOv2 +0.0289 at step 100, +0.0424 at step 500) while moving
it away from the references. The held-out case is unmeasured. A fall here with more training is
overfitting stated at the level of the instrument rather than the judge: the training works and
does not transfer. A rise while the judge falls is a different finding and would need reporting on
its own terms.

Five arms, the same labels and files `arms_embed` used, so the target and reference numbers join
on uid: baseline, LoRA r8 at steps 1000 and 4000, QLoRA r8 at step 2000, soft prompt (iii) at
step 2000.

Instruments are DINOv2-large, CLIP-large and ArcFace, matching `positive_control_check`, so the
memorised and held-out cases are read on the same scales. No API calls.

The two cases are not a controlled comparison: the control ran at 1e-4 on the screen rows, the arms
at 8e-7 on train, at different step counts. The contrast is in direction, not magnitude.

T4. 100 unique targets cached once, then 488 generated images per arm.

## 1. Setup

In [1]:
from google.colab import drive
import os, sys, json, subprocess

drive.mount("/content/drive")
DRIVE  = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
SCORES = f"{DRIVE}/scores"
IMGS, EXPECTED = "/content/imgs", 37063

def n_files(d): return sum(len(f) for _, _, f in os.walk(d)) if os.path.isdir(d) else 0

if n_files(IMGS) < EXPECTED:
    import shutil
    assert shutil.disk_usage("/content").free / 1e9 > 55, "the extracted image set needs ~50 GB"
    os.makedirs(IMGS, exist_ok=True)
    subprocess.run(f'cp "{DRIVE}/imgs_subset.tar" /content/imgs_subset.tar && '
                   f'tar -xf /content/imgs_subset.tar -C {IMGS}', shell=True, check=True)
    assert n_files(IMGS) >= EXPECTED, f"restore incomplete: {n_files(IMGS)} of {EXPECTED}"
print("images:", n_files(IMGS))

if not os.path.exists("/content/test.jsonl"):
    subprocess.run(["cp", f"{DRIVE}/test.jsonl", "/content/test.jsonl"], check=True)
rows = [json.loads(l) for l in open("/content/test.jsonl", encoding="utf-8")]
assert len(rows) == 488, f"expected 488 test rows, got {len(rows)}"

import torch, torch.nn.functional as F, numpy as np
from PIL import Image
from transformers import AutoImageProcessor, AutoModel, CLIPImageProcessor, CLIPVisionModelWithProjection

device = "cuda" if torch.cuda.is_available() else "cpu"
dino_proc = AutoImageProcessor.from_pretrained("facebook/dinov2-large")
dino      = AutoModel.from_pretrained("facebook/dinov2-large").to(device).eval()
clip_proc = CLIPImageProcessor.from_pretrained("openai/clip-vit-large-patch14")
clip      = CLIPVisionModelWithProjection.from_pretrained("openai/clip-vit-large-patch14").to(device).eval()

@torch.no_grad()
def dino_emb(imgs):
    return F.normalize(dino(**dino_proc(images=imgs, return_tensors="pt").to(device)).pooler_output.float(), dim=-1)

@torch.no_grad()
def clip_emb(imgs):
    return F.normalize(clip(**clip_proc(images=imgs, return_tensors="pt").to(device)).image_embeds.float(), dim=-1)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "insightface", "onnxruntime-gpu"], check=False)

# insightface's own downloader does not retry and resolves a different release tag depending on its
# version; arms_embed and positive_control_check pinned v0.7, so the same build is fetched here
IF_DIR = os.path.expanduser("~/.insightface/models")
if not os.path.isdir(f"{IF_DIR}/buffalo_l"):
    os.makedirs(IF_DIR, exist_ok=True)
    subprocess.run(["rm", "-rf", f"{IF_DIR}/buffalo_l", f"{IF_DIR}/buffalo_l.zip"], check=False)
    subprocess.run(["curl", "-L", "--retry", "5", "--retry-all-errors", "--fail",
                    "-o", f"{IF_DIR}/buffalo_l.zip",
                    "https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_l.zip"],
                   check=True)
    subprocess.run(["unzip", "-q", "-o", f"{IF_DIR}/buffalo_l.zip", "-d", f"{IF_DIR}/buffalo_l"], check=True)

from insightface.app import FaceAnalysis
app = FaceAnalysis(name="buffalo_l", providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
app.prepare(ctx_id=0, det_size=(640, 640))

def to_bgr(pil): return np.array(pil.convert("RGB"))[:, :, ::-1]
def largest(faces):
    return max(faces, key=lambda f: (f.bbox[2]-f.bbox[0]) * (f.bbox[3]-f.bbox[1])) if faces else None

print("device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "cpu only")
print("insightface providers:", app.models["recognition"].session.get_providers())

Mounted at /content/drive
images: 37063


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/549 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.22GB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.71GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

[transformers] CLIPVisionModelWithProjection LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
logit_scale                                                  | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.final_layer_norm.weight                           | UNEXPECTED |  | 
text_model.encoder.layers.{

device: cuda | Tesla T4
insightface providers: ['CPUExecutionProvider']


## 2. Arms and target cache

Labels and directories are `arms_embed`'s, so `test__<lab>_identity.jsonl` joins to this notebook's
`test__<lab>_target.jsonl` on uid. `baseline` writes `test_{i:05d}.png` under `gen/baseline`, the
arms write `{i:05d}.png` under `gen/test/<dir>`; both forms are resolved by probing.

The 488 rows carry 100 unique targets, embedded once and shared across the five arms.

In [2]:
ARMS = [
    ("baseline",             "baseline",                   "test__baseline"),
    ("lora_r8_step1000",     "test/lora_r8_step1000",      "test__lora_r8_step1000"),
    ("lora_r8_step4000",     "test/lora_r8",               "test__lora_r8"),
    ("qlora_r8_step2000",    "test/qlora_r8__step2000",    "test__qlora_r8_step2000"),
    ("sp_iii_p139_step2000", "test/sp_iii_p139__step2000", "test__sp_iii_p139_step2000"),
]

def gen_path(gdir, i):
    for pat in (f"{i:05d}.png", f"test_{i:05d}.png"):
        p = f"{DRIVE}/gen/{gdir}/{pat}"
        if os.path.exists(p): return p
    return None

print(f"{'arm':22s} {'gen dir':28s} {'pngs':>5s} {'pattern':>16s} {'gpt':>5s} {'identity':>9s}")
for lab, gdir, sfile in ARMS:
    d = f"{DRIVE}/gen/{gdir}"
    n = len([f for f in os.listdir(d) if f.endswith(".png")]) if os.path.isdir(d) else 0
    p0 = gen_path(gdir, 0)
    pat = os.path.basename(p0).replace("00000", "{i:05d}") if p0 else "MISSING"
    g_ok = os.path.exists(f"{SCORES}/{sfile}.jsonl")
    i_ok = os.path.exists(f"{SCORES}/test__{lab}_identity.jsonl")
    print(f"{lab:22s} {gdir:28s} {n:>5d} {pat:>16s} {'yes' if g_ok else 'NO':>5s} {'yes' if i_ok else 'NO':>9s}")
    assert n >= 488 and p0 and g_ok and i_ok, f"{lab}: incomplete"

uid = lambda r: r["output_image"] + "|" + "|".join(sorted(r["input_images"]))
UID = {i: uid(r) for i, r in enumerate(rows)}
TGT = {i: os.path.join(IMGS, r["output_image"]) for i, r in enumerate(rows)}
uniq = sorted(set(TGT.values()))

import time
t0 = time.time()
if "TEMB" not in dir() or len(TEMB) != len(uniq):
    TEMB = {}
for k, p in enumerate(uniq if len(TEMB) != len(uniq) else []):
    im = Image.open(p).convert("RGB")
    fl = largest(app.get(to_bgr(im)))
    TEMB[p] = (dino_emb([im])[0], clip_emb([im])[0], None if fl is None else fl.normed_embedding)
    if (k + 1) % 25 == 0: print(f"  {k+1}/{len(uniq)}  {time.time()-t0:.0f}s", flush=True)
print(f"target cache: {len(TEMB)} targets "
      f"({sum(1 for v in TEMB.values() if v[2] is not None)} with a detected face) in {time.time()-t0:.0f}s")

arm                    gen dir                       pngs          pattern   gpt  identity
baseline               baseline                       488 test_{i:05d}.png   yes       yes
lora_r8_step1000       test/lora_r8_step1000          488      {i:05d}.png   yes       yes
lora_r8_step4000       test/lora_r8                   488      {i:05d}.png   yes       yes
qlora_r8_step2000      test/qlora_r8__step2000        488      {i:05d}.png   yes       yes
sp_iii_p139_step2000   test/sp_iii_p139__step2000     488      {i:05d}.png   yes       yes
  25/100  73s
  50/100  99s
  75/100  126s
  100/100  148s
target cache: 100 targets (90 with a detected face) in 148s


## 3. Similarity to target

One file per arm, `test__<lab>_target.jsonl`. Save-as-you-go with skip-if-exists on the row index.

In [3]:
def row_target(i, gpath):
    im = Image.open(gpath).convert("RGB")
    gd, gc = dino_emb([im])[0], clip_emb([im])[0]
    gen_embs = [f.normed_embedding for f in app.get(to_bgr(im))]
    td, tc, tf = TEMB[TGT[i]]
    return dict(dino_tgt=float(gd @ td), clip_tgt=float(gc @ tc),
                arc_tgt=(max(float(tf @ ge) for ge in gen_embs) if tf is not None and gen_embs else None),
                gen_faces=len(gen_embs))

# score files are written in thread-completion order, so the row index is the "i" field and never
# the line position
META = {r["i"]: r for r in (json.loads(l) for l in open(f"{SCORES}/test__baseline.jsonl", encoding="utf-8"))}
assert len(META) == len(rows), f"{len(META)} distinct i values, expected {len(rows)}"
assert all(META[i]["uid"] == UID[i] for i in META), "score-file uid does not match test.jsonl at the same i"

for lab, gdir, _ in ARMS:
    out = f"{SCORES}/test__{lab}_target.jsonl"
    done = {json.loads(l)["i"] for l in open(out, encoding="utf-8")} if os.path.exists(out) else set()
    todo = [i for i in range(len(rows)) if i not in done]
    print(f"\n{lab}: {len(done)} done, {len(todo)} to measure", flush=True)
    t0 = time.time()
    with open(out, "a", encoding="utf-8") as f:
        for k, i in enumerate(todo):
            f.write(json.dumps({"i": i, "uid": UID[i], "n_refs": META[i]["n_refs"],
                                "ref_kind": META[i]["ref_kind"], **row_target(i, gen_path(gdir, i))}) + "\n")
            if (k + 1) % 100 == 0:
                f.flush(); print(f"  {k+1}/{len(todo)}  {time.time()-t0:.0f}s", flush=True)
    n = sum(1 for _ in open(out, encoding="utf-8"))
    assert n == len(rows), f"{lab}: {n} rows, expected {len(rows)}"
    print(f"  {lab}: {n} rows in {time.time()-t0:.0f}s")


baseline: 0 done, 488 to measure
  100/488  192s
  200/488  328s
  300/488  481s
  400/488  620s
  baseline: 488 rows in 747s

lora_r8_step1000: 0 done, 488 to measure
  100/488  159s
  200/488  297s
  300/488  446s
  400/488  583s
  lora_r8_step1000: 488 rows in 710s

lora_r8_step4000: 0 done, 488 to measure
  100/488  156s
  200/488  290s
  300/488  430s
  400/488  565s
  lora_r8_step4000: 488 rows in 699s

qlora_r8_step2000: 0 done, 488 to measure
  100/488  167s
  200/488  303s
  300/488  446s
  400/488  599s
  qlora_r8_step2000: 488 rows in 734s

sp_iii_p139_step2000: 0 done, 488 to measure
  100/488  162s
  200/488  307s
  300/488  457s
  400/488  594s
  sp_iii_p139_step2000: 488 rows in 731s


## 4. Target and reference, side by side

Each arm's similarity to the held-out target beside its similarity to the references from
`arms_embed`, and the judge's scores. The reference columns are reproduced rather than recomputed;
they are the same numbers `arms_embed` reported.

In [4]:
import math, statistics as st
from scipy import stats

ov = lambda r: math.sqrt(max(r["PF"], 0) * max(r["SC"], 0))
def load(path, key="i"):
    d = {}
    for l in open(path, encoding="utf-8"):
        r = json.loads(l)
        if "error" not in r: d[r[key]] = r
    return d

TGTS = {lab: load(f"{SCORES}/test__{lab}_target.jsonl")   for lab, _, _ in ARMS}
IDS  = {lab: load(f"{SCORES}/test__{lab}_identity.jsonl") for lab, _, _ in ARMS}
GPT  = {lab: load(f"{SCORES}/{sfile}.jsonl")              for lab, _, sfile in ARMS}
I = sorted(set.intersection(*[set(TGTS[l]) for l, _, _ in ARMS],
                            *[set(IDS[l])  for l, _, _ in ARMS],
                            *[set(GPT[l])  for l, _, _ in ARMS]))
for lab, _, _ in ARMS:
    bad = [i for i in I if TGTS[lab][i]["uid"] != IDS[lab][i]["uid"] or TGTS[lab][i]["uid"] != UID[i]]
    assert not bad, f"{lab}: uid mismatch on rows {bad[:5]}"
print(f"rows common to every file: {len(I)}\n")

KT = ("dino_tgt", "clip_tgt", "arc_tgt")
KR = ("dino", "clip", "arcface_broad")
mean_ok = lambda d, k: (st.mean([d[i][k] for i in I if d[i][k] is not None])
                        if any(d[i][k] is not None for i in I) else float("nan"))

print(f"{'arm':22s} " + " ".join(f"{k:>9s}" for k in KT) + " " +
      " ".join(f"{k+'_ref':>13s}" for k in KR) + f" {'SC':>7s} {'Overall':>8s}")
for lab, _, _ in ARMS:
    print(f"{lab:22s} "
          + " ".join(f"{mean_ok(TGTS[lab], k):>9.4f}" for k in KT) + " "
          + " ".join(f"{mean_ok(IDS[lab], k):>13.4f}" for k in KR)
          + f" {st.mean([GPT[lab][i]['SC'] for i in I]):>7.3f}"
          + f" {st.mean([ov(GPT[lab][i]) for i in I]):>8.3f}")

print(f"\npaired delta against baseline (n={len(I)})")
print(f"{'arm':22s} {'measure':14s} {'delta':>9s} {'SE':>8s} {'p':>10s} {'n':>5s}")
for lab, _, _ in ARMS:
    if lab == "baseline": continue
    for src, keys in ((TGTS, KT), (IDS, KR)):
        for k in keys:
            u = [i for i in I if src[lab][i][k] is not None and src["baseline"][i][k] is not None]
            if len(u) < 10: continue
            d = np.array([src[lab][i][k] - src["baseline"][i][k] for i in u], float)
            se = d.std(ddof=1) / math.sqrt(len(d))
            tag = k if src is TGTS else k + "_ref"
            print(f"{lab:22s} {tag:14s} {d.mean():>+9.4f} {se:>8.4f} "
                  f"{stats.ttest_1samp(d, 0)[1]:>10.3g} {len(u):>5d}")
    d = np.array([GPT[lab][i]["SC"] - GPT["baseline"][i]["SC"] for i in I], float)
    se = d.std(ddof=1) / math.sqrt(len(d))
    print(f"{lab:22s} {'GPT SC':14s} {d.mean():>+9.4f} {se:>8.4f} "
          f"{stats.ttest_1samp(d, 0)[1]:>10.3g} {len(I):>5d}\n")

rows common to every file: 488

arm                     dino_tgt  clip_tgt   arc_tgt      dino_ref      clip_ref arcface_broad_ref      SC  Overall
baseline                  0.7822    0.9109    0.2992        0.4349        0.7605        0.3065   8.088    7.963
lora_r8_step1000          0.7855    0.9149    0.3051        0.4286        0.7599        0.2771   7.846    7.843
lora_r8_step4000          0.7965    0.9237    0.3184        0.4130        0.7522        0.2241   7.432    7.595
qlora_r8_step2000         0.7799    0.9111    0.3058        0.4283        0.7603        0.2826   7.988    7.883
sp_iii_p139_step2000      0.8125    0.9244    0.3751        0.4116        0.7486        0.2170   6.859    7.137

paired delta against baseline (n=488)
arm                    measure            delta       SE          p     n
lora_r8_step1000       dino_tgt         +0.0033   0.0016     0.0378   488
lora_r8_step1000       clip_tgt         +0.0040   0.0010   8.58e-05   488
lora_r8_step1000       arc_tgt 

## 5. Memorised against held-out

The control's rows were trained on, so its movement toward the target is memorisation. These rows
were not, so movement here is generalisation. Read as directions, not magnitudes: the control ran
at 1e-4 on the screen rows at steps 100 and 500, the arms at 8e-7 on train at steps 1000 and 4000.

The null is the identical-config rerun measured on the screen rows, the only split where one was
generated. No equivalent exists for test.

In [5]:
CTRL = {}
for lab in ("baseline", "baseline__rerun", "step100", "step500"):
    p = f"{SCORES}/screen__{lab}_similarity.jsonl"
    if os.path.exists(p): CTRL[lab] = load(p)
print("control similarity files found:", sorted(CTRL))

if {"baseline", "baseline__rerun", "step100", "step500"} <= set(CTRL):
    J = sorted(set.intersection(*[set(v) for v in CTRL.values()]))
    print(f"\nmemorised rows (screen, n={len(J)}), delta against its own baseline:")
    print(f"{'run':16s} " + " ".join(f"{k:>10s}" for k in KT))
    for lab in ("baseline__rerun", "step100", "step500"):
        cells = []
        for k in KT:
            u = [i for i in J if CTRL[lab][i][k] is not None and CTRL["baseline"][i][k] is not None]
            cells.append(f"{st.mean([CTRL[lab][i][k] - CTRL['baseline'][i][k] for i in u]):>+10.4f}"
                         if len(u) > 2 else f"{'n/a':>10s}")
        print(f"{lab:16s} " + " ".join(cells))
else:
    print("\ncontrol similarity files absent; run positive_control_check first for this block")

print(f"\nheld-out rows (test, n={len(I)}), delta against baseline:")
print(f"{'arm':22s} " + " ".join(f"{k:>10s}" for k in KT))
tgt_delta = {}
for lab, _, _ in ARMS:
    if lab == "baseline": continue
    cells = []
    for k in KT:
        u = [i for i in I if TGTS[lab][i][k] is not None and TGTS["baseline"][i][k] is not None]
        m = st.mean([TGTS[lab][i][k] - TGTS["baseline"][i][k] for i in u]) if len(u) > 2 else float("nan")
        tgt_delta[(lab, k)] = m
        cells.append(f"{m:>+10.4f}")
    print(f"{lab:22s} " + " ".join(cells))

print("\nby stratum, LoRA r8 dino_tgt:")
print(f"{'arm':22s} {'stratum':10s} {'n':>4s} {'delta':>10s}")
for lab in ("lora_r8_step1000", "lora_r8_step4000"):
    for rk in ("no_up", "mixed", "all_up"):
        u = [i for i in I if TGTS[lab][i]["ref_kind"] == rk]
        print(f"{lab:22s} {rk:10s} {len(u):>4d} "
              f"{st.mean([TGTS[lab][i]['dino_tgt'] - TGTS['baseline'][i]['dino_tgt'] for i in u]):>+10.4f}")

a, b = tgt_delta[("lora_r8_step1000", "dino_tgt")], tgt_delta[("lora_r8_step4000", "dino_tgt")]
print("\nverdict:")
if b < a < 0:
    print(f"  LoRA moves the held-out outputs away from their targets, further with more training")
    print(f"  ({a:+.4f} at step 1000, {b:+.4f} at step 4000), while the control moved toward the")
    print("  targets it trained on. Training fits its objective and does not transfer: overfitting")
    print("  read off the instrument rather than the judge.")
elif b > a > 0:
    print(f"  LoRA moves the held-out outputs toward their targets, further with more training")
    print(f"  ({a:+.4f} then {b:+.4f}), while the judge's score falls. The training generalises on")
    print("  this measure and the judge disagrees; that disagreement is the result to report.")
else:
    print(f"  Mixed: step 1000 {a:+.4f}, step 4000 {b:+.4f}. Read the table above and the strata")
    print("  before drawing a direction.")

control similarity files found: ['baseline', 'baseline__rerun', 'step100', 'step500']

memorised rows (screen, n=302), delta against its own baseline:
run                dino_tgt   clip_tgt    arc_tgt
baseline__rerun     -0.0002    +0.0006    -0.0001
step100             +0.0289    +0.0150    +0.0814
step500             +0.0424    +0.0132    +0.1134

held-out rows (test, n=488), delta against baseline:
arm                      dino_tgt   clip_tgt    arc_tgt
lora_r8_step1000          +0.0033    +0.0040    +0.0059
lora_r8_step4000          +0.0143    +0.0128    +0.0189
qlora_r8_step2000         -0.0023    +0.0002    +0.0066
sp_iii_p139_step2000      +0.0303    +0.0135    +0.0759

by stratum, LoRA r8 dino_tgt:
arm                    stratum       n      delta
lora_r8_step1000       no_up       157    +0.0029
lora_r8_step1000       mixed       133    +0.0035
lora_r8_step1000       all_up      198    +0.0036
lora_r8_step4000       no_up       157    +0.0080
lora_r8_step4000       mixed      